In [2]:
import sys
sys.path.insert(0, r"C:\Users\mjbou\governance-framework\src")
from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources

log = load_log()
print(f"Log loaded. Rows: {len(log)}")

Log loaded. Rows: 1


## WGI Pipeline

**Source:** World Bank Worldwide Governance Indicators
**Access:** Automated via `wbgapi`
**Download instructions:** See `docs/instructions_data_maintenance.md` — no manual step required.

### Framework usage
| Concept | Role |
|---------|------|
| Government effectiveness and admin quality | Primary tier 1 |
| Political stability and regime durability | Primary tier 1 |
| Regulatory quality | Primary tier 1 |
| Rule of Law category | Cross-check |
| Control of Corruption category | Cross-check |
| Voice and Accountability | Cross-check |

In [16]:
import wbgapi as wb
import pandas as pd
from datetime import datetime

# WGI indicator codes — note GOV_WGI_ prefix required, .EST = estimate (-2.5 to +2.5)
WGI_INDICATORS = {
    'GOV_WGI_GE.EST': 'government_effectiveness',
    'GOV_WGI_PV.EST': 'political_stability',
    'GOV_WGI_RQ.EST': 'regulatory_quality',
    'GOV_WGI_RL.EST': 'rule_of_law',
    'GOV_WGI_CC.EST': 'control_of_corruption',
    'GOV_WGI_VA.EST': 'voice_accountability'
}

# WGI starts in 1996 — override framework start year for this source
WGI_START_YEAR = 1996

# Set WGI as default database
wb.db = 3

print("Fetching WGI data...")
wgi_raw = wb.data.DataFrame(
    list(WGI_INDICATORS.keys()),
    time=range(WGI_START_YEAR, 2026),
    labels=True
)

print(f"Raw shape: {wgi_raw.shape}")
print(wgi_raw.head())

Fetching WGI data...
Raw shape: (1296, 28)
                                      Country  \
economy series                                  
ZWE     GOV_WGI_GE.EST               Zimbabwe   
ZMB     GOV_WGI_GE.EST                 Zambia   
YEM     GOV_WGI_GE.EST            Yemen, Rep.   
PSE     GOV_WGI_GE.EST     West Bank and Gaza   
VIR     GOV_WGI_GE.EST  Virgin Islands (U.S.)   

                                                                   Series  \
economy series                                                              
ZWE     GOV_WGI_GE.EST  Government Effectiveness - Governance estimate...   
ZMB     GOV_WGI_GE.EST  Government Effectiveness - Governance estimate...   
YEM     GOV_WGI_GE.EST  Government Effectiveness - Governance estimate...   
PSE     GOV_WGI_GE.EST  Government Effectiveness - Governance estimate...   
VIR     GOV_WGI_GE.EST  Government Effectiveness - Governance estimate...   

                          YR1996    YR1998    YR2000    YR2002    YR2003 

In [17]:
# Reset index to bring economy and series into columns
wgi = wgi_raw.reset_index()

# Rename columns
wgi = wgi.rename(columns={'economy': 'country_code', 'series': 'indicator_code'})

# Map indicator codes to clean names
wgi['indicator_name'] = wgi['indicator_code'].map(WGI_INDICATORS)

# Melt from wide to long format
year_cols = [c for c in wgi.columns if c.startswith('YR')]
wgi_long = wgi.melt(
    id_vars=['country_code', 'Country', 'indicator_code', 'indicator_name'],
    value_vars=year_cols,
    var_name='year_str',
    value_name='value'
)

# Clean year column
wgi_long['year'] = wgi_long['year_str'].str.replace('YR', '').astype(int)
wgi_long = wgi_long.drop(columns=['year_str', 'indicator_code'])
wgi_long = wgi_long.rename(columns={'Country': 'country_name'})

# Sort
wgi_long = wgi_long.sort_values(['country_code', 'indicator_name', 'year']).reset_index(drop=True)

print(f"Shape: {wgi_long.shape}")
print(f"Years: {wgi_long['year'].min()} — {wgi_long['year'].max()}")
print(f"Countries: {wgi_long['country_code'].nunique()}")
print(f"Indicators: {wgi_long['indicator_name'].unique()}")
print(f"\nMissing values: {wgi_long['value'].isna().sum()} ({wgi_long['value'].isna().mean()*100:.1f}%)")
print(wgi_long.head(10))

Shape: (33696, 5)
Years: 1996 — 2024
Countries: 216
Indicators: <StringArray>
[   'control_of_corruption', 'government_effectiveness',
      'political_stability',       'regulatory_quality',
              'rule_of_law',     'voice_accountability']
Length: 6, dtype: str

Missing values: 1293 (3.8%)
  country_code country_name         indicator_name     value  year
0          ABW        Aruba  control_of_corruption       NaN  1996
1          ABW        Aruba  control_of_corruption       NaN  1998
2          ABW        Aruba  control_of_corruption       NaN  2000
3          ABW        Aruba  control_of_corruption       NaN  2002
4          ABW        Aruba  control_of_corruption       NaN  2003
5          ABW        Aruba  control_of_corruption  0.966286  2004
6          ABW        Aruba  control_of_corruption  1.327120  2005
7          ABW        Aruba  control_of_corruption  1.335303  2006
8          ABW        Aruba  control_of_corruption  1.329441  2007
9          ABW        Aruba  c

In [18]:
# Pivot to wide format — one row per country-year
wgi_wide = wgi_long.pivot_table(
    index=['country_code', 'country_name', 'year'],
    columns='indicator_name',
    values='value'
).reset_index()

# Flatten column names
wgi_wide.columns.name = None
wgi_wide.columns = ['country_code', 'country_name', 'year'] + [f'wgi_{c}' for c in wgi_wide.columns[3:]]

print(f"Shape: {wgi_wide.shape}")
print(f"Columns: {list(wgi_wide.columns)}")
print(f"\nMissing values (%):")
missing_pct = (wgi_wide.isnull().sum() / len(wgi_wide) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(wgi_wide.head())

Shape: (5481, 9)
Columns: ['country_code', 'country_name', 'year', 'wgi_control_of_corruption', 'wgi_government_effectiveness', 'wgi_political_stability', 'wgi_regulatory_quality', 'wgi_rule_of_law', 'wgi_voice_accountability']

Missing values (%):
wgi_government_effectiveness    2.6
wgi_regulatory_quality          2.6
wgi_control_of_corruption       2.0
wgi_political_stability         0.9
wgi_voice_accountability        0.7
wgi_rule_of_law                 0.1
dtype: float64
  country_code country_name  year  wgi_control_of_corruption  \
0          ABW        Aruba  2004                   0.966286   
1          ABW        Aruba  2005                   1.327120   
2          ABW        Aruba  2006                   1.335303   
3          ABW        Aruba  2007                   1.329441   
4          ABW        Aruba  2008                   1.330999   

   wgi_government_effectiveness  wgi_political_stability  \
0                      0.895142                 0.629215   
1              

In [19]:
# Read metadata automatically from source
wgi_source_meta = next(s for s in wb.source.list() if s['id'] == '3')
data_as_of_date = wgi_source_meta['lastupdated'][:7]  # e.g. '2026-03'
latest_year = str(int(wgi_wide['year'].max()))

print(f"Data as of: {data_as_of_date}")
print(f"Latest year in data: {latest_year}")

# Write to processed
output_path = os.path.join(PROCESSED_DIR, "wgi_clean.csv")
wgi_wide.to_csv(output_path, index=False)
print(f"Written: {output_path}")

# Update download log — all values derived from data/metadata, no hardcoding
update_entry(
    "WGI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="wgi_clean.csv",
    latest_available_version=latest_year,
    notes="6 indicators, .EST (estimate) format. Source 3 in wbgapi. Annual from 2003, biennial 1996-2002."
)

print_entry("WGI")

Data as of: 2026-03
Latest year in data: 2024
Written: C:\Users\mjbou\governance-framework\data\processed\wgi_clean.csv
[download_log] Updated entry for WGI
  source_id: WGI
  last_attempted_date: 2026-05-20
  last_successful_download_date: 2026-05-20
  data_as_of_date: 2026-03
  local_filename: wgi_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: 6 indicators, .EST (estimate) format. Source 3 in wbgapi. Annual from 2003, biennial 1996-2002.
